<a href="https://colab.research.google.com/github/FarazHeydar/Prompt-Assembly-for-Healthcare-Applications/blob/main/Coding/02_medsynth_fewshot_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP9800 MSc Project — Dynamic Few-Shot Exemplar Retriever
**Project Title:** Developing Techniques for Prompt Assembly for Healthcare Applications  
**Author:** Faraz | **Supervisor:** Dr. Jinan Fiaidhi  
**Institution:** Lakehead University (F/W 2026–2027)

---

## 1. Overview & Purpose
This standalone notebook implements **Stage 2** of the project pipeline: building the **Dynamic Few-Shot Retrieval Engine** used in **Condition B (Modular Prompt Assembly)**.

Rather than hardcoding static few-shot examples into every prompt, this module:
1. Reads the standardized Parquet datasets generated by **Notebook 01** (`medsynth_fewshot_pool.parquet`, `mts_dialog_benchmark.parquet`, and `custom_experimental_50_cases.parquet`).
2. Encodes the external Few-Shot Pool into dense 384-dimensional semantic vectors using `sentence-transformers/all-MiniLM-L6-v2` on GPU.
3. For every test dialogue in our benchmark and custom 50-case experimental dataset, retrieves the **Top-$k$ ($k=2$) most clinically similar Dialogue-to-SOAP exemplars** via Cosine Similarity:
   $$\text{Cosine Similarity}(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\Vert{}\mathbf{q}\Vert{} \Vert{}\mathbf{d}\Vert{}}$$
4. Exports pre-retrieved Few-Shot bundles (`custom_50_with_fewshot.parquet` and `mts_test_with_fewshot.parquet`) so **Notebook 03 (IBM Granite Pipeline)** can assemble prompts immediately without redundant re-indexing.

## 2. Environment Setup & Automatic Input Discovery from Notebook 01
The cell below installs `sentence-transformers` and automatically locates the `.parquet` files mounted from **Notebook 01** (whether attached via `/kaggle/input/...` on Kaggle or `./processed_data` on Google Colab).

In [ ]:
# ==============================================================================
# UNIVERSAL KAGGLE + GOOGLE COLAB FILE LOADER
# Purpose: Automatically finds input .parquet files on Kaggle (/kaggle/input)
#          OR downloads them automatically from the public GitHub repo when
#          Dr. Jinan opens the notebook in Google Colab!
# ==============================================================================
import urllib.request

def get_project_file(filename: str) -> str:
    """
    1. Checks Kaggle Input & local working directories first.
    2. If running on Google Colab and file is not found locally, automatically
       downloads it from the public GitHub repository!
    """
    search_roots = ["/kaggle/input", "/kaggle/working", "./processed_data", "./retriever_outputs", "./granite_generations", "./data", "."]
    for root in search_roots:
        if os.path.exists(root):
            matches = glob.glob(os.path.join(root, "**", filename), recursive=True)
            if matches:
                print(f"[FOUND LOCALLY] {matches[0]}")
                return matches[0]

    # Fallback for Google Colab: Auto-download from GitHub raw URL
    os.makedirs("./data", exist_ok=True)
    local_dest = os.path.join("./data", filename)
    raw_url = f"https://raw.githubusercontent.com/FarazHeydar/Prompt-Assembly-for-Healthcare-Applications/main/data/{filename}"
    print(f"[COLAB AUTO-DOWNLOAD] Fetching '{filename}' from GitHub ({raw_url})...")
    urllib.request.urlretrieve(raw_url, local_dest)
    print(f"[SUCCESS] Downloaded to {local_dest}")
    return local_dest

## 3. Dense Vector Indexing of the Few-Shot Pool
Here we initialize `all-MiniLM-L6-v2` on the GPU and encode the external Few-Shot Pool dialogues into normalized embeddings. To keep retrieval fast and memory-efficient while maintaining full clinical diversity across ICD-10 categories, we index up to 3,000 external exemplars.

In [ ]:
# ==============================================================================
# DENSE VECTOR ENCODING OF THE FEW-SHOT POOL
# Purpose: Encode the external MedSynth / SOAP pool into L2-normalized vectors
#          for rapid cosine similarity retrieval.
# ==============================================================================

# 1. Sample up to 3,000 exemplars if the pool is massive to ensure fast GPU encoding
if len(df_pool) > 3000:
    df_pool = df_pool.sample(n=3000, random_state=42).reset_index(drop=True)

# 2. Initialize SentenceTransformer on GPU (if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[STATUS] Initializing Embedding Model ('all-MiniLM-L6-v2') on: {device.upper()}")
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# 3. Encode all pool dialogues into L2-normalized dense embeddings
print(f"[STATUS] Encoding {len(df_pool)} Few-Shot Pool dialogues...")
pool_dialogues = df_pool["Dialogue"].astype(str).tolist()
pool_embeddings = embedder.encode(
    pool_dialogues,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"[SUCCESS] Pool Embedding Matrix Shape: {pool_embeddings.shape}")

## 4. Batch Retrieval & Pre-Assembling Few-Shot Exemplars for Notebook 03
In this section, we define `batch_retrieve_exemplars()` and run it across:
1. All **50 cases** in our `Custom Experimental Dataset`.
2. A representative **100-case evaluation split** of `MTS-Dialog` (so **Notebook 03** can benchmark IBM Granite 2B/8B within Kaggle's T4 GPU session limits).

The enriched datasets (with `few_shot_exemplar_1` and `few_shot_exemplar_2` attached to every row) are saved to `/kaggle/working/retriever_outputs` for **Notebook 03**.

In [ ]:
# ==============================================================================
# BATCH FEW-SHOT RETRIEVAL & ARTIFACT EXPORT
# Purpose: Retrieve Top-2 condition-matched SOAP exemplars for every test case
#          and save enriched Parquet files for Notebook 03 (IBM Granite).
# ==============================================================================

def batch_retrieve_exemplars(query_dialogues: list, top_k: int = 2) -> list:
    """
    Encodes a list of query dialogues and retrieves the top_k most similar
    Dialogue-to-SOAP exemplars from the external pool for each query.
    """
    # Encode all queries in one fast GPU batch
    query_vecs = embedder.encode(
        query_dialogues,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    # Compute cosine similarity matrix: shape (num_queries, num_pool)
    sim_matrix = cosine_similarity(query_vecs, pool_embeddings)

    all_retrieved = []
    for i in range(len(query_dialogues)):
        scores = sim_matrix[i]
        top_indices = np.argsort(scores)[::-1][:top_k]

        case_exemplars = []
        for rank, idx in enumerate(top_indices, start=1):
            row = df_pool.iloc[idx]
            case_exemplars.append({
                "rank": rank,
                "similarity": round(float(scores[idx]), 4),
                "dialogue": str(row["Dialogue"]),
                "soap_note": str(row["Note"])
            })
        all_retrieved.append(case_exemplars)
    return all_retrieved

# 1. Retrieve Top-2 Few-Shot Exemplars for all 50 Custom Experimental Cases
print("[1/2] Retrieving Top-2 Few-Shot Exemplars for the 50 Custom Experimental Cases...")
df_custom_50["retrieved_few_shots"] = batch_retrieve_exemplars(
    df_custom_50["dialogue"].astype(str).tolist(), top_k=2
)

# 2. Retrieve Top-2 Few-Shot Exemplars for MTS-Dialog Evaluation Split (N=100)
print("[2/2] Retrieving Top-2 Few-Shot Exemplars for MTS-Dialog Benchmark Split (N=100)...")
df_mts_eval = df_mts.head(100).copy().reset_index(drop=True)
df_mts_eval["retrieved_few_shots"] = batch_retrieve_exemplars(
    df_mts_eval["dialogue"].astype(str).tolist(), top_k=2
)

# 3. Save enriched datasets to Parquet for Notebook 03
custom_out_path = os.path.join(OUTPUT_DIR, "custom_50_with_fewshot.parquet")
mts_out_path    = os.path.join(OUTPUT_DIR, "mts_eval_100_with_fewshot.parquet")

df_custom_50.to_parquet(custom_out_path, index=False)
df_mts_eval.to_parquet(mts_out_path, index=False)

print("=" * 70)
print("[SUCCESS] Exported Enriched Datasets with Few-Shot Exemplars!")
print(f"  -> Saved: {custom_out_path}")
print(f"  -> Saved: {mts_out_path}")
print("=" * 70)

# 4. Inspect the retrieved exemplars for Case EXP_001 (Cardiology)
sample_row = df_custom_50.iloc[0]
print(f"\n[INSPECTION] Retrieved Exemplars for {sample_row['id']} ({sample_row['clinical_domain']}):")
for ex in sample_row["retrieved_few_shots"]:
    print(f"\n  Match #{ex['rank']} (Cosine Similarity: {ex['similarity']})")
    print(f"  Exemplar Dialogue : {ex['dialogue'][:180]}...")
    print(f"  Exemplar SOAP Note: {ex['soap_note'][:180]}...")